# Extract features with ECGFounder

In [8]:
import os
import sys
import torch
import numpy as np

BASE_DIR = os.path.dirname(os.path.abspath('.'))
sys.path.append(BASE_DIR)

In [ ]:
from datautils import create_dataloader

ptbxl_loader = create_dataloader(f"{BASE_DIR}/data/cinc/ptbxl/train.h5",
                                 f"{BASE_DIR}/data/cinc/ptbxl/val.h5",
                                 f"{BASE_DIR}/data/cinc/ptbxl/test.h5",
                                 task="diagnostic_class", batch_size=32)
print("Number of batches [32] in PTB-XL loader:", len(ptbxl_loader))

samitrop_loader = create_dataloader(f"{BASE_DIR}/data/cinc/samitrop/all_data.h5",
                                    task="sex", batch_size=32)
print("Number of batches [32] in SamiTrop loader:", len(samitrop_loader))

Number of batches [32] in PTB-XL loader: 69
Number of batches [32] in SamiTrop loader: 51


In [6]:
from models import ft_12lead_ECGFounder
from utils import set_seed


device = torch.device("cuda:7" if torch.cuda.is_available() else "cpu")
set_seed(42, deterministic=True)

ecgfounder = ft_12lead_ECGFounder(pth=f"{BASE_DIR}/.checkpoints/12_lead_ECGFounder.pth",
                                  n_classes=2, linear_prob=True, device=device)
ecgfounder.return_features = True

Seed set to 42 (deterministic=True)


/home/rafaelsilva/models/cnn-ecg-features/models/finetune_model.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(pth, map_location=device)


In [ ]:
samitrop_features = []

ecgfounder.to(device)
ecgfounder.eval()

with torch.no_grad():

    for batch in samitrop_loader:
        x, y = batch[0].to(device), batch[1].to(device)

        _, deep_feat = ecgfounder(x)

        samitrop_features.append(deep_feat.detach().cpu())

ecgfounder.cpu()

samitrop_features = torch.concat(samitrop_features).numpy()
samitrop_features.shape

(1631, 1024)

In [ ]:
np.save("samitrop_features.npy", samitrop_features)

In [ ]:
ptbxl_features = []

ecgfounder.to(device)
ecgfounder.eval()

with torch.no_grad():

    for batch in ptbxl_loader:
        x, y = batch[0].to(device), batch[1].to(device)

        _, deep_feat = ecgfounder(x)

        ptbxl_features.append(deep_feat.detach().cpu())

ecgfounder.cpu()

ptbxl_features = torch.concat(ptbxl_features).numpy()
ptbxl_features.shape

(2198, 1024)

In [22]:
np.save("ptbxl_test_features.npy", ptbxl_features)